# MP305 Network Flow Models II

In [1]:
from IPython.display import display, Math, Latex

## Overview 

This file contains a number of Python functions for finding the maximal flow through a network $G$ subject to minimal cost using the Ford Fulkerson Algorithm.

The network graph $G$ is stored in a set `G` of two element tuples `(i,j)` describing the directed arcs $(i,j)$ of $G$.

It is assumed that node number $1$ is the source and the greatest node `Nsink` is the sink.  
Thus `G={(1,2),(2,3),(1,3)}`  describes a network with 3 nodes where node 1 is the source and node 3 is the sink.

The capacity $c(i,j)$, flow $\phi(i,j)$ and cost $l(i,j)$ of the arc $(i,j)$ of $G$ are stored in `c[i][j]`, `phi[i][j]` and `l[i][j]`. 
Here `c`, `phi` and `l` are Python lists.

## Python Functions
### The `Initialise(G)` function
Having defined the network $G$, initialise `c`, `phi` and `l` values to zero via the `Initialise` function before defining their values in any particular example.  The global variable `Nsink`,  the sink node of $G$, is also found by the `Initialise` function .

### The main `Iterate(G)`function 
This implements the full algorithm to find the maximum flow with minimal cost. 

## The `Iterate(G)` function is based on a number of other Python functions:

### `Flows(G)` 
This checks for conservation of flow and prints out all of the current flows for G and the total cost of this flow. 

###  `Links(G)`
This finds all arcs `(i,j1)`, ` (i,j2)`,  ... *out* of node `i` of `G`.  The nodes `j1,j2,..` are stored in a global list of sets `Out`.

###  `SourceSink(G)`
This finds all of the paths from source to sink in any network `G` and results in a global set `SinkPaths` of such paths.

###  `IncremNet(G)`
This finds the Incremental Network `Gp` associated with the current flow of the network `G`.

###  `Newflows(G)`
This updates the flows `phi` of `G` according to the best chain found through `Gp`. If the maximal flow is reached, then this is indicated and the maximum flow value is outputed. Otherwise, the output is: the change in flow (`eps`), the cost of the best chain, and the best chain.

###  `Iterate(G)`
Implements the full algorithm to find the maximum flow with minimal cost. 
The output is as follows:

(1) The incremental network `Gp`.

(2) The paths through `Gp` from source to sink. 

(3) The output of `Newflows(G)`. 

(4) The output of `Flows(G)` giving the current flows and cost of `G`.


In [2]:
def Initialise (Gin):
    global c,phi,l,cp,lp,Nsink
    Nsink=1
    for arc in Gin:
        i,j=arc
        Nsink=max(Nsink,i,j)
    # for convenience c[i][j] is capacity of arc [i,j]
    c=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)] 
    phi=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    l=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    cp=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    lp=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    print("All values of c,phi and l initialised to zero")


In [3]:
def Flows (Gin):
    global Nsink,l,phi
    Flowin=[0 for i in range(Nsink+1)]
    Flowout=[0 for i in range(Nsink+1)]
    for arc in Gin:
        i,j=arc
        Flowin[j] = Flowin[j] + phi[i][j]
        Flowout[i] = Flowout[i] + phi[i][j]
    for k in range(2,Nsink):
        if Flowin[k] != Flowout[k]:
            print("*** ERROR *** Flow not conserved at node", k)
    if Flowout[1] != Flowin[Nsink]:
        print("*** ERROR *** Flow not conserved at source or sink")
    Totalcost = 0
    for arc in Gin:
        i,j=arc
        phi_ij = phi[i][j]
        Totalcost = Totalcost + l[i][j]*phi_ij
        print(arc," has flow ",phi_ij)
    print("Total Cost is ", Totalcost)


In [4]:
def Links (Gin):
    global Nsink,Out
    Out=[set() for k in range(Nsink)] # labelled 0..Nsink-1
    for arc in Gin:
        i,j = arc
        Out[i - 1] = Out[i - 1] | set([j])

In [5]:
def SourceSink(Gin):
# finds all paths SinkPaths from source 1 to sink Nsink of network G 
    global Nsink,SinkPaths
    Links(Gin)
    Paths = set() # current paths from source stored as set of tuples
    SinkPaths = set() # paths from source to sink Nsink stored as set of tuples
    path = 1 # source node label
    for node in Out[0]:# need out edge from node 1
        pathn = (path,node)
        if node == Nsink:
            SinkPaths = SinkPaths | set([pathn])
        else:
            Paths = Paths | set([pathn])
    Npaths = len(Paths)
    while (0 < Npaths):
        NewPaths = set()
        for oldpath in Paths:
            nold = len(oldpath)
            m = oldpath[-1] # last node in tuple oldpath
            for mout in Out[m-1]:
                if not mout in oldpath:
                    if mout == Nsink:
                        SinkPaths = SinkPaths | set([oldpath+tuple([Nsink])])
                    else:
                        NewPaths = NewPaths | set([oldpath+tuple([mout])])
        Paths = NewPaths
        Npaths = len(Paths)
    print("Paths from source to sink: ",SinkPaths)

In [6]:
def Newflows(Gin):
# A procedure to modify original flows on Gin along SinkPaths of Gp with minimal cost
    global Gp,phi,c,l,cp,lp,ArcSign,Out
    SourceSink(Gp)
    if SinkPaths == set():
        Links(Gin)
        Flow = 0
        for node in Out[0]:
            Flow = Flow + phi[1][node]
        Cost=0
        for arc in Gin:
            i,j=arc
            Cost=Cost+l[i][j]*phi[i][j]
        print("Maximal flow found:", Flow, " with minimal cost ", Cost)
    else:
        for k in range(len(SinkPaths)):
            cost = 0
            epset = set()
            path=list(SinkPaths)[k]
            for n in range(0, len(path)-1):
                i = path[n]; j = path[n+1];  epset = epset | set([cp[i][j]]); cost = lp[i][j] + cost
            eps = min(tuple(epset))
            if k == 0: # first path
                mincost = cost; bestpath = path; besteps = eps
            elif cost < mincost: 
                mincost = cost; bestpath = path; besteps = eps
        print("A best path in Gp is ", bestpath, " of minimum cost ", mincost)
        print("The min capacity on this path is epsilon ", besteps)
        print("The min cost is ", mincost)
        for k in range(0, len(bestpath) - 1):
            i = bestpath[k]; j = bestpath[k+1]
            if ArcSign[i][j] == 1:
                phinewij = phi[i][j] + besteps; phi[i][j]=phinewij
            else:
                phinewji=phi[j][i] = phi[j][i] - besteps; phi[j][i]=phinewji
    #print("Flow=",Flow)

In [7]:
def IncremNet(Gin):
# procedure to create incremental network Gp from given flow network G 
    global Gp,Nsink,phi,c,l,cp,lp,ArcSign
# define lists for ArcSign, cp and lp  (indexed by 0..Nsink-1)
    cp=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    lp=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    ArcSign=[[0 for i in range(Nsink+1)] for j in range(Nsink+1)]
    Gp=set([])
    for arc in Gin:
        i,j=arc
        pij = phi[i][j]; pji = phi[j][i]; cij = c[i][j]; lij = l[i][j]
        if (pij < cij and (pji == 0 or not (j,i) in Gin)): # ij arc
            #Gp edges, capacitites and costs added
            cpij = cij - pij; cp[i][j] = cpij; lpij = lij; lp[i][j] =lpij
            ArcSign[i][j] = 1
            Gp=Gp | {(i,j)}
        if pij>0: # ji arc
            cpji = pij; cp[j][i] = cpji; lpji=-lij; lp[j][i] = lpji
            ArcSign[j][i] = -1
            Gp=Gp | {(j,i)}
    print("Incremental Network:",Gp)

In [8]:
def Iterate(Gin):
    IncremNet(Gin)
    Newflows(Gin)
    for arc in Gin:
        i,j=arc
        print((i,j)," flow = ", phi[i][j])

# Q 1. Power Station/Coal Field Supply
### Find the maximal flow for minimal cost for the network below where (capacity,cost) is shown:

![Network](Lab2_1.jpg)


## This is the Coalfield/Power station supply problem discussed in the notes
### * Find the incremental network, its capacities and costs at each iteration.

In [9]:
G={(1,2),(1,3),(2,4),(2,5),(3,4),(3,5),(4,6),(5,6)}

In [10]:
Initialise(G)

All values of c,phi and l initialised to zero


## Input capacities

In [11]:
c[1][2]=3; c[1][3]=5; c[2][4]=3; c[2][5]=3 
c[3][5]=5; c[3][4]=5; c[4][6]=2; c[5][6]=5

## Input costs

In [12]:
l[1][2]=5; l[1][3]=3; 
l[2][4]=3 
l[2][5]=6; l[3][4]=5 
l[3][5]=9 

In [13]:
Flows(G)

(2, 4)  has flow  0
(1, 2)  has flow  0
(3, 4)  has flow  0
(4, 6)  has flow  0
(5, 6)  has flow  0
(2, 5)  has flow  0
(1, 3)  has flow  0
(3, 5)  has flow  0
Total Cost is  0


In [14]:
Iterate(G)

Incremental Network: {(2, 4), (1, 2), (2, 5), (3, 4), (5, 6), (4, 6), (1, 3), (3, 5)}
Paths from source to sink:  {(1, 2, 5, 6), (1, 3, 4, 6), (1, 2, 4, 6), (1, 3, 5, 6)}
A best path in Gp is  (1, 3, 4, 6)  of minimum cost  8
The min capacity on this path is epsilon  2
The min cost is  8
(2, 4)  flow =  0
(1, 2)  flow =  0
(3, 4)  flow =  2
(4, 6)  flow =  2
(5, 6)  flow =  0
(2, 5)  flow =  0
(1, 3)  flow =  2
(3, 5)  flow =  0


### Continue to iterate until max flow for minimal cost found

In [15]:
Iterate(G)

Incremental Network: {(2, 4), (1, 2), (3, 4), (4, 3), (3, 1), (6, 4), (5, 6), (2, 5), (1, 3), (3, 5)}
Paths from source to sink:  {(1, 2, 5, 6), (1, 2, 4, 3, 5, 6), (1, 3, 5, 6)}
A best path in Gp is  (1, 2, 5, 6)  of minimum cost  11
The min capacity on this path is epsilon  3
The min cost is  11
(2, 4)  flow =  0
(1, 2)  flow =  3
(3, 4)  flow =  2
(4, 6)  flow =  2
(5, 6)  flow =  3
(2, 5)  flow =  3
(1, 3)  flow =  2
(3, 5)  flow =  0


In [16]:
Iterate(G)

Incremental Network: {(2, 4), (2, 1), (3, 4), (4, 3), (6, 5), (3, 1), (6, 4), (5, 6), (1, 3), (3, 5), (5, 2)}
Paths from source to sink:  {(1, 3, 5, 6)}
A best path in Gp is  (1, 3, 5, 6)  of minimum cost  12
The min capacity on this path is epsilon  2
The min cost is  12
(2, 4)  flow =  0
(1, 2)  flow =  3
(3, 4)  flow =  2
(4, 6)  flow =  2
(5, 6)  flow =  5
(2, 5)  flow =  3
(1, 3)  flow =  4
(3, 5)  flow =  2


In [17]:
Iterate(G)

Incremental Network: {(2, 4), (2, 1), (3, 4), (4, 3), (6, 5), (3, 1), (6, 4), (5, 3), (1, 3), (3, 5), (5, 2)}
Paths from source to sink:  set()
Maximal flow found: 7  with minimal cost  73
(2, 4)  flow =  0
(1, 2)  flow =  3
(3, 4)  flow =  2
(4, 6)  flow =  2
(5, 6)  flow =  5
(2, 5)  flow =  3
(1, 3)  flow =  4
(3, 5)  flow =  2


# Q2. (*)  
## A road network is shown below with the capacity and time taken per car on each road indicated.
![Network](Lab2_2.jpg)


## * Find the maximal flow through the network for minimal total travel time for all cars from A to B. 
## * Compare this to the flow from B to A.

### Note: You can use the network and capacitites you created in the first lab Network Flows I, Question 3.

## #Initialise the Bidirectional Graph A -> B

In [26]:
G = {(1,2),(1,3),(2,1),(2,5),(3,1),(3,4),(3,7),(4,2),(4,3),(4,6),(5,6),(5,8),(6,4),(7,6),(7,8),(8,5),(8,7)}
Initialise(G)

All values of c,phi and l initialised to zero


In [27]:
c[1][2] = 5; c[1][3] = 4; c[2][1] = 5 
c[2][5] = 3; c[3][1] = 4; c[3][4] = 5  
c[3][7] = 3; c[4][2] = 4; c[4][3] = 5
c[4][6] = 5; c[5][6] = 2; c[5][8] = 6
c[6][4] = 5; c[7][6] = 4; c[7][8] = 4; 
c[8][5] = 6; c[8][7] = 4; 

l[1][2] = 3; l[1][3] = 2; l[2][1] = 3 
l[2][5] = 2; l[3][1] = 2; l[3][4] = 1  
l[3][7] = 2; l[4][2] = 3; l[4][3] = 1
l[4][6] = 2; l[5][6] = 3; l[5][8] = 3
l[6][4] = 2; l[7][6] = 3; l[7][8] = 1; 
l[8][5] = 3; l[8][7] = 1; 

Flows(G)

(1, 2)  has flow  0
(2, 1)  has flow  0
(3, 4)  has flow  0
(4, 3)  has flow  0
(3, 1)  has flow  0
(3, 7)  has flow  0
(4, 6)  has flow  0
(6, 4)  has flow  0
(7, 6)  has flow  0
(2, 5)  has flow  0
(1, 3)  has flow  0
(5, 8)  has flow  0
(8, 7)  has flow  0
(4, 2)  has flow  0
(5, 6)  has flow  0
(8, 5)  has flow  0
(7, 8)  has flow  0
Total Cost is  0


## *Maximum Capacity = 6 & Minimal Cost = 39

In [ ]:
#Running 100 iterations of the Iterate algorithm in a while loop to get output in one cell
i = 100
while (i > 0):
    Iterate(G)
    i = i - 1

Incremental Network: {(1, 2), (2, 1), (3, 4), (4, 3), (3, 1), (4, 6), (6, 4), (7, 3), (7, 6), (1, 3), (5, 2), (5, 8), (8, 7), (4, 2), (5, 6), (8, 5), (7, 8)}
Paths from source to sink:  set()
Maximal flow found: 6  with minimal cost  39
(1, 2)  flow =  3
(2, 1)  flow =  0
(3, 4)  flow =  0
(4, 3)  flow =  0
(3, 1)  flow =  0
(3, 7)  flow =  3
(4, 6)  flow =  0
(6, 4)  flow =  0
(7, 6)  flow =  0
(2, 5)  flow =  3
(1, 3)  flow =  3
(5, 8)  flow =  3
(8, 7)  flow =  0
(4, 2)  flow =  0
(5, 6)  flow =  0
(8, 5)  flow =  0
(7, 8)  flow =  3
Incremental Network: {(1, 2), (2, 1), (3, 4), (4, 3), (3, 1), (4, 6), (6, 4), (7, 3), (7, 6), (1, 3), (5, 2), (5, 8), (8, 7), (4, 2), (5, 6), (8, 5), (7, 8)}
Paths from source to sink:  set()
Maximal flow found: 6  with minimal cost  39
(1, 2)  flow =  3
(2, 1)  flow =  0
(3, 4)  flow =  0
(4, 3)  flow =  0
(3, 1)  flow =  0
(3, 7)  flow =  3
(4, 6)  flow =  0
(6, 4)  flow =  0
(7, 6)  flow =  0
(2, 5)  flow =  3
(1, 3)  flow =  3
(5, 8)  flow =  3
(8, 

## #Initialise the Bidirectional Graph B -> A

In [30]:
G = {(1,2),(1,3),(2,1),(2,4),(3,1),(3,4),(4,5),(5,4),(5,6),(5,7),(6,3),(6,8),(7,2),(7,5),(7,8),(8,6),(8,7)}
Initialise(G)
Flows(G)

All values of c,phi and l initialised to zero
(2, 4)  has flow  0
(1, 2)  has flow  0
(2, 1)  has flow  0
(3, 4)  has flow  0
(3, 1)  has flow  0
(5, 4)  has flow  0
(5, 7)  has flow  0
(8, 6)  has flow  0
(1, 3)  has flow  0
(8, 7)  has flow  0
(6, 8)  has flow  0
(4, 5)  has flow  0
(5, 6)  has flow  0
(7, 2)  has flow  0
(7, 5)  has flow  0
(6, 3)  has flow  0
(7, 8)  has flow  0
Total Cost is  0


In [31]:
c[1][2] = 4; c[1][3] = 6; c[2][1] = 4 
c[2][4] = 4; c[3][1] = 6; c[3][4] = 2  
c[4][5] = 5; c[5][4] = 5; c[5][6] = 4
c[5][7] = 5; c[6][3] = 3; c[6][8] = 5
c[7][2] = 3; c[7][5] = 5; c[7][8] = 4; 
c[8][6] = 5; c[8][7] = 4; 

l[1][2] = 1; l[1][3] = 3; l[2][1] = 1 
l[2][4] = 3; l[3][1] = 3; l[3][4] = 3  
l[4][5] = 2; l[5][4] = 2; l[5][6] = 3
l[5][7] = 1; l[6][3] = 2; l[6][8] = 3
l[7][2] = 2; l[7][5] = 1; l[7][8] = 2 
l[8][6] = 3; l[8][7] = 2; 

## *Maximal Flow = 5 & Minimal Cost = 50

In [ ]:
#Running 100 iterations of the Iterate algorithm in a while loop to get output in one cell
i = 100
while (i > 0):
    Iterate(G)
    i = i - 1

Incremental Network: {(3, 4), (2, 1), (4, 3), (3, 1), (6, 5), (6, 8), (5, 4), (8, 7), (5, 7), (4, 2), (5, 6), (8, 6), (7, 2), (7, 5), (6, 3), (1, 3)}
Paths from source to sink:  set()
Maximal flow found: 5  with minimal cost  50
(2, 4)  flow =  4
(1, 2)  flow =  4
(2, 1)  flow =  0
(3, 4)  flow =  1
(3, 1)  flow =  0
(5, 4)  flow =  0
(5, 7)  flow =  4
(8, 6)  flow =  0
(1, 3)  flow =  1
(8, 7)  flow =  0
(6, 8)  flow =  1
(4, 5)  flow =  5
(5, 6)  flow =  1
(7, 2)  flow =  0
(7, 5)  flow =  0
(6, 3)  flow =  0
(7, 8)  flow =  4
Incremental Network: {(3, 4), (2, 1), (4, 3), (3, 1), (6, 5), (6, 8), (5, 4), (8, 7), (5, 7), (4, 2), (5, 6), (8, 6), (7, 2), (7, 5), (6, 3), (1, 3)}
Paths from source to sink:  set()
Maximal flow found: 5  with minimal cost  50
(2, 4)  flow =  4
(1, 2)  flow =  4
(2, 1)  flow =  0
(3, 4)  flow =  1
(3, 1)  flow =  0
(5, 4)  flow =  0
(5, 7)  flow =  4
(8, 6)  flow =  0
(1, 3)  flow =  1
(8, 7)  flow =  0
(6, 8)  flow =  1
(4, 5)  flow =  5
(5, 6)  flow =  1
(7

 # Q.3 (*) Soft Drinks Stock Control
 ## A soft drinks firm buys fruit at the beginning of each month $i$ at a cost per 100kg $p_i$ in units of 1000 Euro. 
 ## The firm can store up 2000kg of fruit at any given time where cost in units 1000 Euro of refrigeration per month per 100kg is $r_i$. 
## The consumption requirements are $c_i$ per month in 100kg units. 

## Based on last year's figures,  the following estimates have been made:
\begin{array}{|l|l|l|l|l|l|l|l|l|l|l|l|l|}
\hline
i & Jan & Feb & Mar & April & May & June & July & Aug & Sept & Oct & Nov & 
Dec \\ \hline
p_{i} & 18 & 17 & 17 & 15 & 12 & 8 & 7 & 6 & 9 & 12 & 14 & 17 \\ \hline
r_{i} & 1 & 1 & 2 & 2 & 3 & 5 & 6 & 6 & 5 & 3 & 2 & 1 \\ \hline
c_{i} & 9 & 6 & 6 & 7 & 11 & 14 & 16 & 18 & 15 & 10 & 7 & 6 \\ \hline
\end{array}

## (a) Find the best purchasing schedule starting from January based on these estimates assuming that the firm has no fruit in storage on Jan 1st. 
## How much will fruit cost for the year and how much will be spent on refrigeration?

## (b) Suppose that the firm has 500kg of fruit in storage at the beginning of the year.
## What is the best purchasing schedule that ensures that 500kg of fruit is again in storage at the very end of the year?

### Task 3a: (a) Generate a graph with source as 1 and sink as 14. All the months from Jan-Dec are marked as : 2 - 13   

In [34]:
# Source = 1; Jan-Dec = 2 -> 13; Sink = 14
G = {(1,2),(1,3),(1,4),(1,5),(1,6),(1,7),(1,8),(1,9),(1,10),(1,11),(1,12),(1,13),
     (2,3),(2,14),(3,4),(3,14),(4,5),(4,14),(5,6),(5,14),(6,7),(6,14),(7,8),(7,14),(8,9),(8,14),(9,10),(9,14),(10,11),(10,14),(11,12),(11,14),(12,13),(12,14),(13,14)}
Initialise(G)
Flows(G)

All values of c,phi and l initialised to zero
(3, 4)  has flow  0
(12, 13)  has flow  0
(8, 9)  has flow  0
(9, 14)  has flow  0
(1, 6)  has flow  0
(11, 14)  has flow  0
(1, 3)  has flow  0
(1, 9)  has flow  0
(2, 14)  has flow  0
(1, 12)  has flow  0
(13, 14)  has flow  0
(6, 14)  has flow  0
(4, 5)  has flow  0
(5, 6)  has flow  0
(4, 14)  has flow  0
(9, 10)  has flow  0
(8, 14)  has flow  0
(1, 2)  has flow  0
(10, 11)  has flow  0
(1, 5)  has flow  0
(1, 11)  has flow  0
(10, 14)  has flow  0
(1, 8)  has flow  0
(6, 7)  has flow  0
(12, 14)  has flow  0
(3, 14)  has flow  0
(5, 14)  has flow  0
(1, 4)  has flow  0
(2, 3)  has flow  0
(11, 12)  has flow  0
(1, 7)  has flow  0
(1, 13)  has flow  0
(1, 10)  has flow  0
(7, 8)  has flow  0
(7, 14)  has flow  0
Total Cost is  0


### Capacities and Cost matrix initialization

In [35]:
#Capacity at start of each month from the source would be infinite as we can purchase at any limit
c[1][2] = float('inf'); c[1][3] = float('inf');c[1][4] = float('inf')
c[1][5] = float('inf'); c[1][6] = float('inf'); c[1][7] = float('inf')
c[1][8] = float('inf'); c[1][9] = float('inf'); c[1][10] = float('inf')
c[1][11] = float('inf'); c[1][12] = float('inf'); c[1][13] = float('inf')

#Capacity from each month to next month would be 20 Units (which equates to 2000 kg) as its the maximum storage capacity
c[2][3] = 20; c[3][4] = 20; c[4][5] = 20
c[5][6] = 20; c[6][7] = 20; c[7][8] = 20
c[8][9] = 20; c[9][10] = 20; c[10][11] = 20
c[11][12] = 20; c[12][13] = 20

#Capacity from each month to sink would be consumption of that month
c[2][14] = 9; c[3][14] = 6; c[4][14] = 6
c[5][14] = 7; c[6][14] = 11; c[7][14] = 14
c[8][14] = 16; c[9][14] = 18; c[10][14] = 15
c[11][14] = 10; c[12][14] = 7; c[13][14] = 6
 
#Costs from Source to each month would equal Pi (Purchasing Units) of each month
l[1][2] = 18; l[1][3] = 17; l[1][4] = 17
l[1][5] = 15; l[1][6] = 12; l[1][7] = 8
l[1][8] = 7; l[1][9] = 6; l[1][10] = 9
l[1][11] = 12; l[1][12] = 14; l[1][13] = 17

#Costs from each month to next month would be equal to refrigerration costs of that month
l[2][3] = 1; l[3][4] = 1; l[4][5] = 2
l[5][6] = 2; l[6][7] = 3; l[7][8] = 5
l[8][9] = 6; l[9][10] = 6; l[10][11] = 5
l[11][12] = 3; l[12][13] = 2

#Costs from each month to sink would be 0 as there is no cost for consumption of drinks ;)
l[2][14] = 0; l[3][14] = 0; l[4][14] = 0
l[5][14] = 0; l[6][14] = 0; l[7][14] = 0
l[8][14] = 0; l[9][14] = 0; l[10][14] = 0
l[11][14] = 0; l[12][14] = 0; l[13][14] = 0

### After considerable Iterations we get the cost as 1384 and maximal flow as 125

In [37]:
i = 200
while (i > 0):
    Iterate(G)
    i = i - 1
Iterate(G)

Incremental Network: {(12, 1), (3, 4), (14, 4), (3, 1), (14, 7), (12, 13), (5, 1), (14, 10), (14, 13), (8, 9), (1, 6), (1, 3), (1, 9), (1, 12), (7, 1), (14, 6), (4, 5), (14, 3), (14, 9), (5, 6), (14, 12), (9, 1), (9, 10), (11, 1), (1, 2), (10, 11), (2, 1), (1, 5), (1, 11), (6, 1), (1, 8), (6, 7), (14, 2), (4, 1), (14, 5), (14, 11), (14, 8), (8, 1), (10, 1), (1, 4), (2, 3), (11, 12), (1, 7), (1, 13), (13, 12), (1, 10), (7, 8)}
Paths from source to sink:  set()
Maximal flow found: 125  with minimal cost  1384
(3, 4)  flow =  0
(12, 13)  flow =  6
(8, 9)  flow =  0
(9, 14)  flow =  18
(1, 6)  flow =  11
(11, 14)  flow =  10
(1, 3)  flow =  6
(1, 9)  flow =  18
(2, 14)  flow =  9
(1, 12)  flow =  13
(13, 14)  flow =  6
(6, 14)  flow =  11
(4, 5)  flow =  0
(5, 6)  flow =  0
(4, 14)  flow =  6
(9, 10)  flow =  0
(8, 14)  flow =  16
(1, 2)  flow =  9
(10, 11)  flow =  0
(1, 5)  flow =  7
(1, 11)  flow =  10
(10, 14)  flow =  15
(1, 8)  flow =  16
(6, 7)  flow =  0
(12, 14)  flow =  7
(3, 14)

### Task 3(b): Generate a graph with source as 1, a new node as 2, months jan-dec as 3-14 and sink as 15

In [38]:
# Source = 1; new nodes = 2; Jan-Dec = 3 -> 14; Sink = 15 [Note: new node won't have a consumtion unit to sink]
G = {(1,2),(1,3),(1,4),(1,5),(1,6),(1,7),(1,8),(1,9),(1,10),(1,11),(1,12),(1,13),(1,14),
     (2,3),(3,4),(3,15),(4,5),(4,15),(5,6),(5,15),(6,7),(6,15),(7,8),(7,15),(8,9),(8,15),(9,10),(9,15),(10,11),(10,15),(11,12),(11,15),(12,13),(12,15),(13,14),(13,15),(14,15)}
Initialise(G)
Flows(G)

All values of c,phi and l initialised to zero
(6, 15)  has flow  0
(3, 4)  has flow  0
(12, 13)  has flow  0
(4, 15)  has flow  0
(8, 9)  has flow  0
(1, 6)  has flow  0
(8, 15)  has flow  0
(1, 3)  has flow  0
(1, 9)  has flow  0
(10, 15)  has flow  0
(1, 12)  has flow  0
(13, 14)  has flow  0
(4, 5)  has flow  0
(5, 6)  has flow  0
(12, 15)  has flow  0
(3, 15)  has flow  0
(14, 15)  has flow  0
(9, 10)  has flow  0
(5, 15)  has flow  0
(1, 2)  has flow  0
(10, 11)  has flow  0
(1, 5)  has flow  0
(1, 11)  has flow  0
(1, 8)  has flow  0
(1, 14)  has flow  0
(6, 7)  has flow  0
(7, 15)  has flow  0
(1, 4)  has flow  0
(9, 15)  has flow  0
(2, 3)  has flow  0
(11, 12)  has flow  0
(1, 7)  has flow  0
(1, 13)  has flow  0
(11, 15)  has flow  0
(1, 10)  has flow  0
(13, 15)  has flow  0
(7, 8)  has flow  0
Total Cost is  0


In [39]:
#Capacity at start of each month from the source would be infinite
c[1][2] = 5; c[1][3] = float('inf');c[1][4] = float('inf')
c[1][5] = float('inf'); c[1][6] = float('inf'); c[1][7] = float('inf')
c[1][8] = float('inf'); c[1][9] = float('inf'); c[1][10] = float('inf')
c[1][11] = float('inf'); c[1][12] = float('inf'); c[1][13] = float('inf');  c[1][14] = float('inf')

#Capacity from each month to next month would be 20 Units (which equates to 2000 kg)
c[2][3] = 5; c[3][4] = 20; c[4][5] = 20
c[5][6] = 20; c[6][7] = 20; c[7][8] = 20
c[8][9] = 20; c[9][10] = 20; c[10][11] = 20
c[11][12] = 20; c[12][13] = 20; c[13][14] = 20

#Capacity from each month to sink would be consumption of that month
c[3][15] = 9; c[4][15] = 6; c[5][15] = 6
c[6][15] = 7; c[7][15] = 11; c[8][15] = 14
c[9][15] = 16; c[10][15] = 18; c[11][15] = 15
c[12][15] = 10; c[13][15] = 7; c[14][15] = 11
 
#Costs from Source to each month would equal Pi (Purchasing Units) of each month
l[1][2] = 0; l[1][3] = 18; l[1][4] = 17
l[1][5] = 17; l[1][6] = 15; l[1][7] = 12
l[1][8] = 8; l[1][9] = 7; l[1][10] = 6
l[1][11] = 9; l[1][12] = 12; l[1][13] = 14; l[1][14] = 17

#Costs from each month to next month would be equal to refrigerration costs of that month
l[2][3] = 0; l[3][4] = 1; l[4][5] = 1
l[5][6] = 2; l[6][7] = 2; l[7][8] = 3 
l[8][9] = 5; l[9][10] = 6; l[10][11] = 6
l[11][12] = 5;l[12][13] = 3; l[13][14] = 2; 

#Costs from each month to sink would be 0
l[3][15] = 0; l[4][15] = 0; l[5][15] = 0
l[6][15] = 0; l[7][15] = 0; l[8][15] = 0
l[9][15] = 0; l[10][15] = 0; l[11][15] = 0
l[12][15] = 0; l[13][15] = 0; l[14][15] = 0

### After considerable Iterations we get the cost as 1374 and maximal flow as 130

In [41]:
i = 200
while (i > 0):
    Iterate(G)
    i = i - 1
Iterate(G)

Incremental Network: {(12, 1), (3, 4), (3, 1), (12, 13), (5, 1), (14, 13), (8, 9), (1, 6), (1, 3), (1, 9), (1, 12), (13, 14), (15, 5), (7, 1), (15, 8), (15, 14), (15, 11), (4, 5), (5, 6), (9, 1), (9, 10), (11, 1), (10, 11), (2, 1), (13, 1), (1, 5), (15, 4), (6, 1), (1, 11), (1, 8), (1, 14), (15, 7), (15, 13), (15, 10), (6, 7), (3, 2), (4, 1), (8, 1), (10, 1), (1, 4), (11, 12), (1, 7), (15, 6), (1, 13), (15, 3), (15, 9), (1, 10), (15, 12), (7, 8)}
Paths from source to sink:  set()
Maximal flow found: 130  with minimal cost  1374
(6, 15)  flow =  7
(3, 4)  flow =  0
(12, 13)  flow =  0
(4, 15)  flow =  6
(8, 9)  flow =  0
(1, 6)  flow =  7
(8, 15)  flow =  14
(1, 3)  flow =  4
(1, 9)  flow =  16
(10, 15)  flow =  18
(1, 12)  flow =  10
(13, 14)  flow =  11
(4, 5)  flow =  0
(5, 6)  flow =  0
(12, 15)  flow =  10
(3, 15)  flow =  9
(14, 15)  flow =  11
(9, 10)  flow =  0
(5, 15)  flow =  6
(1, 2)  flow =  5
(10, 11)  flow =  0
(1, 5)  flow =  6
(1, 11)  flow =  15
(1, 8)  flow =  14
(1, 1